# Serving the Forecast Model on a T4

Load test of the production XGBoost model on the CPU and on the GPU,
with and without batching
([docs/gpu-serving.md](../docs/gpu-serving.md)). The changes and their
success criteria were written before this notebook was run.

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The
notebook uses one GPU and takes about 20 minutes.

It needs `ml/artifacts/serving_inputs/` from
`scripts/research/export_serving_inputs.py`, so upload the repository
with that folder as a Kaggle dataset named `earth-intelligence` and add
it to the notebook.

Output: `/kaggle/working/serving.zip`. Unzip it into
`ml/artifacts/serving/`.

In [ ]:
!pip install -q nvidia-ml-py 2>&1 | tail -3

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPOSITORY = "https://github.com/khoahonguyen0210/Earth_intelligence.git"
INPUT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/earth-intelligence")
OUTPUT = Path("/kaggle/working/serving")


def uploaded_project():
    for marker in INPUT.rglob("ml/pyproject.toml"):
        return marker.parent.parent
    for archive in INPUT.rglob("earth-intelligence.tar.gz"):
        shutil.unpack_archive(archive, PROJECT)
        return PROJECT
    return None


if not PROJECT.exists():
    source = uploaded_project()
    if source is None:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY, str(PROJECT)], check=True
        )
    elif source != PROJECT:
        shutil.copytree(source, PROJECT)

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=index,name,memory.total,clocks.max.sm,power.limit --format=csv | tee {OUTPUT}/gpus.csv
!nproc
!python -c "import xgboost; print(xgboost.__version__, xgboost.build_info().get('USE_CUDA'))"

## Load test

Measures accuracy and scoring times, then runs each setup at rising
request rates until it misses the service level.

In [ ]:
!cd {PROJECT} && PYTHONPATH=ml/src python scripts/research/run_serving_benchmark.py --inputs ml/artifacts/serving_inputs --output {OUTPUT} 2>&1 | tee {OUTPUT}/log.txt

In [ ]:
shutil.make_archive("/kaggle/working/serving", "zip", OUTPUT)